# Experiment 3 — `spark.sql.shuffle.partitions`

One workload, one setting changed per run. The workload is a window function, which shuffles every row.
Spark UI: http://localhost:4040 → **Stages**: task count, *Shuffle Read* summary (min / median / max), *Spill (memory)* / *Spill (disk)*, duration.

In [2]:
import time
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("m3-exp3-shuffle-partitions").getOrCreate()
big = spark.read.parquet("/opt/data/m3/orders_big")

w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
first_orders = big.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1)

def run(label, n_partitions, coalescing):
    spark.conf.set("spark.sql.shuffle.partitions", str(n_partitions))
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", str(coalescing).lower())
    t0 = time.perf_counter()
    n = first_orders.count()
    print(f"{label}: shuffle.partitions={n_partitions}, AQE coalescing={coalescing}: {n:,} rows in {time.perf_counter() - t0:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 19:44:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
ranked = big.withColumn("rn", F.row_number().over(w))   # no filter: every row, every column shuffled

def run(label, n_partitions, coalescing):
    spark.conf.set("spark.sql.shuffle.partitions", str(n_partitions))
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", str(coalescing).lower())
    t0 = time.perf_counter()
    ranked.write.format("noop").mode("overwrite").save()
    print(f"{label}: shuffle.partitions={n_partitions}, AQE coalescing={coalescing}: {time.perf_counter() - t0:.1f}s")

## Run A — far too few, AQE coalescing off

**PREDICT:** tasks in the stage after the shuffle; shuffle read per task; spill or not; rough duration.

In [4]:
run("A", 4, coalescing=False)

[Stage 3:=============================>                             (2 + 2) / 4]

A: shuffle.partitions=4, AQE coalescing=False: 14.6s


## Run B — far too many, AQE coalescing off

**PREDICT:** the same four things, relative to A.

In [5]:
run("B", 2000, coalescing=False)

[Stage 6:===================================================> (1941 + 9) / 2000]

B: shuffle.partitions=2000, AQE coalescing=False: 8.0s


## Run C — the same 2000, AQE coalescing on

**PREDICT:** how many tasks AQE ends up with, and why that number.

In [6]:
run("C", 2000, coalescing=True)

[Stage 9:==============================================>          (17 + 4) / 21]

C: shuffle.partitions=2000, AQE coalescing=True: 5.2s


## Run D — the same 4, AQE coalescing on

**PREDICT:** does AQE change anything this time?

In [7]:
run("D", 4, coalescing=True)

[Stage 12:>                                                         (0 + 4) / 4]

D: shuffle.partitions=4, AQE coalescing=True: 10.1s


## Run E — a value you choose

From the total shuffle size seen in A/B and a target per-task size, pick a value, say why in chat, then run it with coalescing off.

In [8]:
MY_VALUE = 21  # <- set this
run("E", MY_VALUE, coalescing=False)

[Stage 15:==========================================>             (16 + 5) / 21]

E: shuffle.partitions=21, AQE coalescing=False: 5.4s


## Stop

In [ ]:
spark.stop()